# Reconciliation audit: where and why the two sources disagree

Every player-game both sources report is compared on thirteen values (ADR 0005). This notebook
reads the result tables for each league: how much of each season the second source covers, how
often the sources agree, and which values differ when they do. A rising conflict rate is the first
sign one source changed shape.

In [ ]:
import sys
sys.path.insert(0, str(__import__('pathlib').Path.cwd()))
import matplotlib.pyplot as plt
import pandas as pd
from fs import connect, describe, leagues, save_findings

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
con = connect()
LEAGUES = leagues(con)
LABEL = {'wnba': 'WNBA', 'nba': 'NBA'}
COLOR = {'wnba': '#b23a0a', 'nba': '#1a7f4b'}
findings = {}
print(describe(con), 'Leagues:', ', '.join(LABEL[lg] for lg in LEAGUES))

In [ ]:
summary = con.sql('''
    select league, season, player_games, played_player_games, played_matched,
           round(100 * played_match_rate, 2) as played_match_pct,
           exact, minor, conflict, round(100 * coalesce(conflict_rate, 0), 3) as conflict_pct
    from reconciliation_summary order by league, season
''').df()
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for lg in LEAGUES:
    part = summary[summary['league'] == lg]
    axes[0].plot(part['season'], part['played_match_pct'], marker='o', label=LABEL[lg], color=COLOR[lg])
    axes[1].plot(part['season'], part['conflict_pct'], marker='o', label=LABEL[lg], color=COLOR[lg])
axes[0].axhline(99, ls='--', color='grey', lw=1)
axes[0].set_title('Played player-games matched (%)')
axes[0].legend()
axes[1].set_title('Conflict rate among matched (%)')
for ax in axes:
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## Which values disagree

`reconciliation` holds one row per differing value with both figures and the resolution (the
league value wins; the second source fills in only where the league is null). Did-not-play lines
and minutes rounding are left out, so what remains is a genuine disagreement.

In [ ]:
by_stat = con.sql('''
    select league, stat, count(*) as differences, round(avg(abs(delta)), 2) as mean_abs_delta,
           sum(case when resolution = 'espn' then 1 else 0 end) as filled_from_second_source
    from reconciliation group by 1, 2 order by 1, 3 desc
''').df()
by_stat

In [ ]:
for lg in LEAGUES:
    part = summary[summary['league'] == lg]
    latest = part.iloc[-1]
    stats = by_stat[by_stat['league'] == lg]
    worst = part.sort_values('played_match_pct').iloc[0]
    findings[lg] = {
        'headline': f"{latest['played_match_pct']:.1f}% of {latest['season']} played player-games are matched across sources, and {latest['conflict_pct']:.2f}% of those disagree on a counting stat.",
        'facts': [
            {'label': 'Seasons compared', 'value': f"{len(part)} ({part['season'].iloc[0]} to {part['season'].iloc[-1]})"},
            {'label': 'Lowest coverage', 'value': f"{worst['season']}: {worst['played_match_pct']:.1f}%"},
            {'label': 'Player-games in conflict, all seasons', 'value': f"{int(part['conflict'].sum()):,} of {int(part['played_matched'].sum()):,}"},
            {'label': 'Most-disputed value', 'value': f"{stats.iloc[0]['stat']} ({int(stats.iloc[0]['differences'])} differences)" if len(stats) else 'none'},
        ],
        'table': {
            'columns': ['Season', 'Matched (played)', 'Conflict rate'],
            'rows': [[r['season'], f"{r['played_match_pct']:.1f}%", f"{r['conflict_pct']:.2f}%"] for _, r in part.iterrows()],
        },
    }
save_findings('02_reconciliation_audit', {'title': 'Reconciliation audit', 'leagues': findings})
findings